In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchrl.envs import GymEnv
from torchrl.modules import QValueActor, MLP
from torchrl.data import TensorDictReplayBuffer, LazyTensorStorage
from torchrl.objectives import DQNLoss, SoftUpdate
from torchrl.collectors import SyncDataCollector



In [2]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")

# 1. Environment
env = GymEnv("CartPole-v1", device=device)



Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
# 2. Q-network
# make torch nn module
class net(torch.nn.Module):
    def __init__(self, input_size=1, hidden_size=20, output_size=1):
        super().__init__()
        self.fc1 = torch.nn.Linear(input_size, hidden_size)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(hidden_size, output_size)
        self.hiddenlayer = torch.nn.Linear(hidden_size, hidden_size)
        self.sigmoid = torch.nn.Sigmoid()
        
        
        self._init_weights_gaussian()
    
    def _init_weights_xavier(self):
        torch.nn.init.xavier_uniform_(self.fc1.weight)
        torch.nn.init.zeros_(self.fc1.bias)
        torch.nn.init.xavier_uniform_(self.fc2.weight)
        torch.nn.init.zeros_(self.fc2.bias)

    def _init_weights_gaussian(self):
        torch.nn.init.normal_(self.fc1.weight, mean=0, std=0.1)
        torch.nn.init.zeros_(self.fc1.bias)
        torch.nn.init.normal_(self.fc2.weight, mean=0, std=0.1)
        torch.nn.init.zeros_(self.fc2.bias)

    def _init_weights_kaiming(self):
        torch.nn.init.kaiming_uniform_(self.fc1.weight, nonlinearity='relu')
        torch.nn.init.zeros_(self.fc1.bias)
        torch.nn.init.kaiming_uniform_(self.fc2.weight, nonlinearity='relu')
        torch.nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        x = self.fc1(x)
        x = self.sigmoid(x)
        x = self.fc2(x)
        return x
    
qnet = net(input_size=4, hidden_size=128, output_size=2).to(device)

In [4]:
actor = QValueActor(qnet, spec=env.action_spec).to(device)


In [5]:
# 3. Replay Buffer
replay_buffer = TensorDictReplayBuffer(
    storage=LazyTensorStorage(10000),
    batch_size=64,
)



In [6]:
# 4. Data Collector
collector = SyncDataCollector(
    env,
    actor,
    frames_per_batch=1000,
    total_frames=50000,
    device=device,
)



e:\Doc\MLenv\lib\site-packages\torchrl\collectors\_base.py:1188: DeprecationWarning: SyncDataCollector has been deprecated and will be removed in v0.13. Please use Collector instead.
  warnings.warn(


In [7]:
# 5. Loss + Target network
loss_module = DQNLoss(actor, action_space=env.action_spec)
target_updater = SoftUpdate(loss_module, eps=0.995)

optimizer = optim.Adam(loss_module.parameters(), lr=1e-3)


In [8]:
# 6. Training Loop
for i, data in enumerate(collector):

    # store transitions
    replay_buffer.extend(data)

    if len(replay_buffer) < 1000:
        continue

    # sample batch
    batch = replay_buffer.sample()

    # compute loss
    loss = loss_module(batch)
    loss_value = loss["loss"]

    optimizer.zero_grad()
    loss_value.backward()
    optimizer.step()

    target_updater.step()

    if i % 10 == 0:
        print(f"Iteration {i}, Loss: {loss_value.item():.4f}")

2026-05-17 01:36:54,567 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([10000]) shape [END]
Iteration 0, Loss: 0.8868
Iteration 10, Loss: 0.1016
Iteration 20, Loss: 0.0704
Iteration 30, Loss: 0.0489
Iteration 40, Loss: 0.0524


In [9]:
# 7. Run the trained policy
with torch.no_grad():
    # Reset the environment and run a rollout
    rollout = env.rollout(max_steps=1000, policy=actor)
    total_reward = rollout["next", "reward"].sum().item()
    print(f"Rollout complete. Total reward: {total_reward}")

    # Optional: Display the first few transitions
    print("First 5 observations:\n", rollout["observation"][:5])
    print("First 5 actions:\n", rollout["action"][:5])


Rollout complete. Total reward: 10.0
First 5 observations:
 tensor([[-0.0302, -0.0482, -0.0304,  0.0259],
        [-0.0312, -0.2429, -0.0299,  0.3089],
        [-0.0360, -0.4376, -0.0237,  0.5920],
        [-0.0448, -0.6324, -0.0119,  0.8771],
        [-0.0574, -0.8273,  0.0057,  1.1660]])
First 5 actions:
 tensor([[1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0]])
